# Lab 3.2 - Amazon SageMaker: Exploring Data

**Educate edition.** Replaces `en_us/3_2-machinelearning.ipynb`.

## Objectives
* Explore and display statistics using pandas
* Use charts to explore data characteristics
* Look for correlation between features

**Cost note:** no AWS services used beyond the notebook instance.

**Prerequisite:** run Lab 3.1 first so `vertebral_column.csv` exists.

In [ ]:
import warnings; warnings.simplefilter('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

df = pd.read_csv('vertebral_column.csv')
print('Loaded', df.shape)
df.head()

## Step 1 - Structure of the dataset

Always start with shape, dtypes and null counts. `info()` gives all
three at once.

In [ ]:
df.info()

### Target distribution

This dataset is **imbalanced**: roughly 2 abnormal patients for every
1 normal one. That matters later - a model that predicts "Abnormal"
for everyone would already be about 68% accurate, so accuracy alone
will be a misleading metric in Lab 3.6.

In [ ]:
counts = df['class'].value_counts()
print(counts)
print()
print('Proportions:')
print((counts / len(df)).round(3))
print()
print('Baseline accuracy of always predicting the majority class: '
      f'{counts.max()/len(df):.1%}')

counts.plot(kind='bar', title='Class distribution', rot=0)
plt.ylabel('count'); plt.show()

## Step 2 - Summary statistics

`describe()` reports count, mean, std, min, quartiles and max for each
numeric feature.

Look closely at `degree_spondylolisthesis`: its max is far above the
75th percentile. That is an outlier worth noting.

In [ ]:
df.describe().T.round(2)

## Step 3 - Distribution of each feature

Histograms show the shape of each feature. Most are roughly normal;
`degree_spondylolisthesis` is heavily right-skewed.

In [ ]:
features = [c for c in df.columns if c != 'class']

df[features].hist(figsize=(12, 8), bins=30, edgecolor='black')
plt.tight_layout(); plt.show()

### Box plots split by class

This is the most useful chart in the lab. Where the two boxes barely
overlap, that feature separates the classes well and will be
predictive.

Note how `degree_spondylolisthesis` and `pelvic_incidence` separate the
classes far better than `pelvic_radius` does.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(data=df, x='class', y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

## Step 4 - Outliers

The single extreme `degree_spondylolisthesis` value is a genuine
recorded measurement, not a data-entry error, so we keep it. XGBoost is
tree-based and therefore robust to outliers - it splits on rank, not on
magnitude. A linear model would need this handled.

In [ ]:
col = 'degree_spondylolisthesis'
q1, q3 = df[col].quantile([0.25, 0.75])
iqr = q3 - q1
hi = q3 + 1.5 * iqr
out = df[df[col] > hi]
print(f'{col}: IQR upper fence = {hi:.1f}')
print(f'{len(out)} rows above the fence')
out.sort_values(col, ascending=False).head()

## Step 5 - Correlation between features

Strongly correlated features carry redundant information.

`pelvic_incidence` is correlated with both `sacral_slope` and
`pelvic_tilt` - and that is expected, because anatomically
**pelvic_incidence = sacral_slope + pelvic_tilt**. The dataset contains
a near-exact linear dependency.

In [ ]:
corr = df[features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, cbar_kws={'shrink': .8})
plt.title('Feature correlation'); plt.tight_layout(); plt.show()

In [ ]:
check = (df['sacral_slope'] + df['pelvic_tilt'] - df['pelvic_incidence']).abs()
print('Max deviation from pelvic_incidence = sacral_slope + pelvic_tilt:',
      round(check.max(), 6))

### Pair plot

A pair plot shows every pairwise scatter plus each feature's
distribution, coloured by class. It is the fastest way to see which
feature *combinations* separate the classes.

In [ ]:
sns.pairplot(df, hue='class', height=1.8, plot_kws={'s': 18, 'alpha': .7})
plt.show()

## Conclusion

You have:
* Examined the structure of the dataset with pandas
* Viewed statistics and distributions with pandas and Matplotlib
* Found correlation between features, including an exact anatomical
  dependency

Key findings to carry forward:
* The target is imbalanced (about 68% Abnormal) - use more than accuracy
* `degree_spondylolisthesis` is the strongest single separator
* `pelvic_incidence` is a linear combination of two other features

**Remember to Stop the notebook instance when you are done.**

Next: `3_3-machinelearning.ipynb`